# Benchmark RAL — MULTI-VIEW WEIGHTED 7:3 (study-level), CXR vs CXR+EHR

Biến thể multi-view **trộn CÓ TRỌNG SỐ** thay vì concat: mỗi study lấy
`img = (0.7·frontal + 0.3·lateral) / (0.7·hf + 0.3·hl)` → **768-dim** (chuẩn hóa theo view có mặt) + 2 cờ `has_frontal/has_lateral`.
- Study chỉ 1 view → dùng riêng view đó (chuẩn hóa nên trọng số về 1).
- Dùng **dataset viewfix** (đã khôi phục view cho ảnh ViewPosition=NaN).
- **1 study = 1 mẫu**, 13 nhãn, subject 70/10/20, default+tuned-F2, RAL default (γ⁺0,γ⁻4,τ.05,λ1.5).
- Có **resume** (FRESH flag). Output file MỚI trong WRITE_DIR.

## 0. Cài CatBoost

In [ ]:
!pip -q install catboost

## 1. Dựng dataset MULTI-VIEW WEIGHTED 7:3 (study-level)

In [ ]:
import os, numpy as np, pandas as pd
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e: print('drive:', e)
WRITE_DIR='/content/drive/MyDrive/lpft_out'
SRC_DIR  ='/content/drive/MyDrive/MIMIC_CXR/planB_out'
CHEX13=['Atelectasis','Cardiomegaly','Consolidation','Edema','Enlarged Cardiomediastinum',
 'Fracture','Lung Lesion','Lung Opacity','No Finding','Pleural Effusion',
 'Pleural Other','Pneumonia','Pneumothorax']
WF,WL=0.7,0.3                                    # trọng số frontal : lateral

MV=os.path.join(WRITE_DIR,'fusion_dataset_lpft_mv73_viewfix.parquet')   # file MỚI
if os.path.exists(MV):
    df=pd.read_parquet(MV); print('nạp lại mv73 có sẵn:', df.shape)
else:
    df0=pd.read_parquet(os.path.join(WRITE_DIR,'fusion_dataset_lpft_fullmicro_viewfix.parquet'))  # viewfix
    cm =pd.read_parquet(os.path.join(SRC_DIR,'manifest_cohort.parquet'))
    cmr=cm[['dicom_id']+CHEX13].drop_duplicates('dicom_id').rename(columns={c:'raw_'+c for c in CHEX13})
    df0=df0.merge(cmr,on='dicom_id',how='left')
    IMG0=[f'img_{i}' for i in range(768)]; nonimg=[c for c in df0.columns if c not in IMG0 and c!='study_id']
    front=df0[df0.view=='frontal'].groupby('study_id')[IMG0].mean()      # mean feature frontal/study
    lat  =df0[df0.view=='lateral'].groupby('study_id')[IMG0].mean()
    meta =df0.groupby('study_id')[nonimg].first()                        # subject_id, y_, raw_, EHR
    studies=meta.index
    Ff=front.reindex(studies); hf=Ff.notna().all(1).astype(float).values; Ff=Ff.fillna(0.0).values
    Ll=lat.reindex(studies);   hl=Ll.notna().all(1).astype(float).values; Ll=Ll.fillna(0.0).values
    num=WF*Ff*hf[:,None] + WL*Ll*hl[:,None]                              # trộn có trọng số
    den=(WF*hf + WL*hl)[:,None]                                          # chuẩn hóa theo view có mặt
    IMGw=(num/den).astype('float32')                                     # (studies, 768)
    df=meta.copy(); df[IMG0]=IMGw; df=df.reset_index()
    df['has_frontal']=hf.astype(int); df['has_lateral']=hl.astype(int)
    df.to_parquet(MV,index=False)

RAW=['raw_'+c for c in CHEX13]
both=int((df.has_frontal&df.has_lateral).sum())
print('mv73 study-level:', df.shape, '| studies:', len(df),
      '| có CẢ 2 view:', both, f'({100*both/len(df):.1f}%)',
      '| chỉ frontal:', int((df.has_frontal.astype(bool)&~df.has_lateral.astype(bool)).sum()),
      '| chỉ lateral:', int((~df.has_frontal.astype(bool)&df.has_lateral.astype(bool)).sum()))

## 2. Split subject 70/10/20 + preprocess (IMG = img_ 768 + cờ)

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
rng=np.random.default_rng(42); uniq=np.unique(df.subject_id.values); rng.shuffle(uniq)
n=len(uniq); s1,s2=int(.70*n),int(.80*n)
trs,vas,tes=set(uniq[:s1]),set(uniq[s1:s2]),set(uniq[s2:])
sp=np.where(df.subject_id.isin(trs),'train',np.where(df.subject_id.isin(vas),'val','test'))
tr=sp=='train'; va=sp=='val'; te=sp=='test'
assert not (trs&tes) and not (trs&vas) and not (vas&tes)
print('subject 70/10/20 (study):', {k:int((sp==k).sum()) for k in ['train','val','test']})

IMG=[f'img_{i}' for i in range(768)]+['has_frontal','has_lateral']   # 770
DROP={'dicom_id','subject_id','study_id','hadm_id','stay_id','first_hadm_id','first_admittime',
      'first_icd10_codes','study_dt','admittime','ed_intime','encounter_time','split','view',
      'disease_group','chiefcomplaint'}
if df['gender'].dtype=='object': df['gender']=df['gender'].map({'M':1,'F':0})
EHR=[c for c in df.columns if not c.startswith(('img_','y_','dx_','raw_','has_')) and c not in DROP]
FEATSETS={'CXR':IMG, 'CXR+EHR':IMG+EHR}
print('IMG(mv73):',len(IMG),'| EHR:',len(EHR))
def prep(cols):
    X=df[cols].apply(pd.to_numeric,errors='coerce').to_numpy('float32')
    X=SimpleImputer(strategy='median').fit(X[tr]).transform(X)
    return StandardScaler().fit(X[tr]).transform(X).astype('float32')
Xcache={k:prep(v) for k,v in FEATSETS.items()}
RAWm=df[RAW].to_numpy('float32')
Yuo=np.where(np.isnan(RAWm),0.,np.where(RAWm==-1,1.,RAWm)).astype('float32')   # U-Ones train
print({k:v.shape for k,v in Xcache.items()})

## 3. RAL + train 4 model (có RESUME) + eval

In [ ]:
import torch, torch.nn as nn
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,fbeta_score,
                             average_precision_score,roc_auc_score)
from tqdm.auto import tqdm
import warnings; warnings.filterwarnings('ignore')
dev=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GP,GN,CLIP,LAM,EPS=0.0,4.0,0.05,1.5,1e-8
def ral_loss(z,y):
    p=torch.sigmoid(z)
    Lp=y*(1-p).clamp(min=EPS).pow(GP)*(-torch.log(p.clamp(min=EPS)))
    pt=(p-CLIP).clamp(min=0); Ln=(LAM-p)*(1-y)*pt.pow(GN)*(-torch.log((1-pt).clamp(min=EPS)))
    return Lp+Ln
def ral_gh(margin,y):
    z=torch.tensor(np.asarray(margin,dtype=np.float64),requires_grad=True); yt=torch.tensor(np.asarray(y,dtype=np.float64))
    L=ral_loss(z,yt).sum(); g,=torch.autograd.grad(L,z,create_graph=True); h,=torch.autograd.grad(g.sum(),z)
    return g.detach().numpy(), np.clip(h.detach().numpy(),1e-6,None)
def obj_xgb(y_true,y_pred): return ral_gh(y_pred,y_true)
def obj_lgb(y_true,y_pred): return ral_gh(y_pred,y_true)
class RALObjCat:
    def calc_ders_range(self,approxes,targets,weights):
        g,h=ral_gh(np.array(approxes),np.array(targets)); d1=-g; d2=-h
        if weights is not None: w=np.array(weights); d1=d1*w; d2=d2*w
        return list(zip(d1.tolist(),d2.tolist()))
sig=lambda z:1/(1+np.exp(-np.clip(z,-30,30)))
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

SAMPLE_TRAIN=None
tr_idx=np.where(tr)[0]; va_idx=np.where(va)[0]; te_idx=np.where(te)[0]
if SAMPLE_TRAIN and len(tr_idx)>SAMPLE_TRAIN: tr_idx=np.random.default_rng(0).choice(tr_idx,SAMPLE_TRAIN,replace=False)
def macro_ap(S,R):
    aps=[]
    for j in range(13):
        col=R[:,j]; k=col!=-1; yt=np.where(np.isnan(col[k]),0.,col[k]).astype(int)
        if yt.sum()>0: aps.append(average_precision_score(yt,S[k,j]))
    return float(np.mean(aps))
def train_boost(kind,X):
    Sv=np.zeros((len(va_idx),13)); St=np.zeros((len(te_idx),13))
    for j in tqdm(range(13),desc=kind,leave=False):
        ytr=Yuo[tr_idx,j]
        if kind=='XGBoost':
            m=XGBClassifier(n_estimators=300,max_depth=4,learning_rate=0.1,tree_method='hist',objective=obj_xgb,n_jobs=-1).fit(X[tr_idx],ytr)
            mv=m.predict(X[va_idx],output_margin=True); mt=m.predict(X[te_idx],output_margin=True)
        elif kind=='LightGBM':
            m=LGBMClassifier(n_estimators=300,learning_rate=0.1,num_leaves=31,objective=obj_lgb,n_jobs=-1,verbose=-1).fit(X[tr_idx],ytr)
            mv=m.predict(X[va_idx],raw_score=True); mt=m.predict(X[te_idx],raw_score=True)
        else:
            m=CatBoostClassifier(iterations=300,depth=4,learning_rate=0.1,loss_function=RALObjCat(),eval_metric='AUC',verbose=0).fit(X[tr_idx],ytr)
            mv=m.predict(X[va_idx],prediction_type='RawFormulaVal'); mt=m.predict(X[te_idx],prediction_type='RawFormulaVal')
        Sv[:,j]=sig(mv); St[:,j]=sig(mt)
    return Sv,St
def train_mlp(X):
    d=X.shape[1]
    net=nn.Sequential(nn.Linear(d,512),nn.ReLU(),nn.Dropout(0.3),nn.Linear(512,256),nn.ReLU(),nn.Dropout(0.3),nn.Linear(256,13)).to(dev)
    opt=torch.optim.AdamW(net.parameters(),lr=1e-3,weight_decay=1e-2)
    Xt=torch.tensor(X); Yt=torch.tensor(Yuo); best=-1; bs=None; bad=0
    for ep in range(40):
        net.train(); perm=np.random.default_rng(ep).permutation(tr_idx)
        for i in range(0,len(perm),4096):
            b=perm[i:i+4096]; opt.zero_grad(); loss=ral_loss(net(Xt[b].to(dev)),Yt[b].to(dev)).mean(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad(): Sv=torch.sigmoid(net(Xt[va_idx].to(dev))).cpu().numpy()
        ap=macro_ap(Sv,RAWm[va_idx])
        if ap>best: best=ap; bs={k:v.detach().cpu().clone() for k,v in net.state_dict().items()}; bad=0
        else:
            bad+=1
            if bad>=6: break
    net.load_state_dict(bs); net.eval()
    with torch.no_grad():
        Sv=torch.sigmoid(net(Xt[va_idx].to(dev))).cpu().numpy(); St=torch.sigmoid(net(Xt[te_idx].to(dev))).cpu().numpy()
    return Sv,St
GRID=np.linspace(0.02,0.98,49)
def best_thr(y,s): f=[fbeta_score(y,(s>=t).astype(int),beta=2,zero_division=0) for t in GRID]; return float(GRID[int(np.argmax(f))])
def metr(y,pred): return dict(accuracy=accuracy_score(y,pred),precision=precision_score(y,pred,zero_division=0),
    recall=recall_score(y,pred,zero_division=0),f1=f1_score(y,pred,zero_division=0),f2=fbeta_score(y,pred,beta=2,zero_division=0))

MODELS=['XGBoost','LightGBM','CatBoost','MLP']
CKPT=os.path.join(WRITE_DIR,'benchmark_ral_mv73_viewfix.csv')    # CSV MỚI
FRESH=True                                    # True vì lần đầu chạy trên mv73 viewfix
if FRESH and os.path.exists(CKPT): os.remove(CKPT)
done=set(zip(pd.read_csv(CKPT).featset,pd.read_csv(CKPT).model)) if os.path.exists(CKPT) else set()
if done: print('đã xong (bỏ qua):',done)
for fs in FEATSETS:
    X=Xcache[fs]
    for mname in MODELS:
        if (fs,mname) in done: print('bỏ qua',fs,mname); continue
        Sv,St=train_mlp(X) if mname=='MLP' else train_boost(mname,X)
        rows=[]
        for j,c in enumerate(CHEX13):
            rv=RAWm[va_idx,j]; kv=rv!=-1; yv=np.where(np.isnan(rv[kv]),0.,rv[kv]).astype(int); sv=Sv[kv,j]
            rt=RAWm[te_idx,j]; kt=rt!=-1; yt=np.where(np.isnan(rt[kt]),0.,rt[kt]).astype(int); st=St[kt,j]
            au=roc_auc_score(yt,st) if 0<yt.sum()<len(yt) else np.nan
            ap=average_precision_score(yt,st) if yt.sum()>0 else np.nan
            tthr=best_thr(yv,sv)
            for mode,pred,thr in [('default',(st>=0.5).astype(int),0.5),('tuned',(st>=tthr).astype(int),tthr)]:
                rows.append(dict(model=mname,featset=fs,label=c,thr_mode=mode,thr=round(float(thr),4),
                    **metr(yt,pred), auprc=ap, auroc=au, prevalence=round(float(yt.mean()),4), n_pos=int(yt.sum())))
        pd.DataFrame(rows).to_csv(CKPT,mode='a',header=not os.path.exists(CKPT),index=False)
        print(f'✓ {fs} {mname} | auprc(mean)={np.nanmean([r["auprc"] for r in rows]):.3f} | f2 tuned={np.mean([r["f2"] for r in rows if r["thr_mode"]=="tuned"]):.3f}')
res=pd.read_csv(CKPT); print('\nrows:',len(res))

## 4. Heatmap per model (CXR vs CXR+EHR) + Δ 6 chỉ số

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
res=pd.read_csv(os.path.join(WRITE_DIR,'benchmark_ral_mv73_viewfix.csv'))
METRICS=['accuracy','precision','recall','f1','f2','auprc']; MODELS_O=['XGBoost','LightGBM','CatBoost','MLP']
def hm(sub,ax,title):
    m=sub.set_index('label')[METRICS].reindex(CHEX13)
    sns.heatmap(m,ax=ax,cmap='RdYlGn',vmin=0,vmax=1,annot=True,fmt='.3f',linewidths=.5,linecolor='black',annot_kws={'size':7},cbar_kws={'label':'score'})
    ax.set_title(title); ax.set_xlabel(''); ax.set_ylabel(''); ax.set_yticklabels(ax.get_yticklabels(),rotation=0)
for mname in MODELS_O:
    fig,ax=plt.subplots(1,2,figsize=(16,6))
    for a,fs in zip(ax,['CXR','CXR+EHR']):
        hm(res[(res.model==mname)&(res.featset==fs)&(res.thr_mode=='tuned')],a,f'{mname} · {fs} · RAL MV7:3 tuned')
    plt.tight_layout(); plt.show()
for metric in METRICS:
    D=pd.DataFrame(index=CHEX13)
    for mname in MODELS_O:
        a=res[(res.model==mname)&(res.featset=='CXR')    &(res.thr_mode=='tuned')].set_index('label')[metric].reindex(CHEX13)
        b=res[(res.model==mname)&(res.featset=='CXR+EHR')&(res.thr_mode=='tuned')].set_index('label')[metric].reindex(CHEX13)
        D[mname]=b-a
    plt.figure(figsize=(7,6))
    sns.heatmap(D,cmap='RdBu_r',center=0,vmin=-.15,vmax=.15,annot=True,fmt='+.3f',linewidths=.5,linecolor='black',annot_kws={'size':8},cbar_kws={'label':f'Δ {metric}'})
    plt.title(f'Δ (CXR+EHR − CXR) · {metric} · RAL MV7:3 (đỏ=EHR giúp)'); plt.ylabel(''); plt.yticks(rotation=0); plt.tight_layout(); plt.show()
print('== MACRO (mean 13 nhãn) =='); print(res.groupby(['model','featset','thr_mode'])[METRICS].mean().round(4))